In [8]:
#let's build a simple tokennizer first
text="I love machine learning"

#let's split the word
tokens=text.split();

#step 2->create vocabulary
vocab ={
    word:idx
    for idx,word in enumerate(set(tokens))
}
#step 3; let's convert token's in to id
token_id=[vocab[word] for word in tokens]

#let 's get words back
print("Tokens:", tokens)
print("Vocabulary:", vocab)
print("Token IDs:", token_id)

reversing_vocab={v: k for k,v in vocab.items()}
decoded_tokens=" ".join(reversing_vocab[idx] for idx in token_id)

print("decoded:",decoded_tokens)

Tokens: ['I', 'love', 'machine', 'learning']
Vocabulary: {'love': 0, 'machine': 1, 'I': 2, 'learning': 3}
Token IDs: [2, 0, 1, 3]
decoded: I love machine learning


## Byte Pair Encoding (BPE) — Quick Notes

### 1. Why BPE Exists
* **Word-level tokenizers** explode vocabulary size, struggle with unseen words (`<UNK>`), and don't share subword structure (`play`, `playing`, `player`).
* **Character-level tokenizers** keep vocabularies tiny, but create **long sequences** $\rightarrow$ higher attention compute ($O(n^2)$) and harder sequence modeling.
* **BPE (subword tokenization)** sits in between: frequent words stay whole tokens, while rare/new words split into known subwords.

### 2. How BPE Training Works
1. **Initialize:** Split words into characters (plus an end-of-word marker like `</w>` or byte-level symbols).
2. **Count pairs:** Tally adjacent symbol pairs weighted by word frequency in the corpus (`freq += corpus[word]`).
3. **Merge top pair:** Merge the most frequent adjacent pair into a new symbol and record the merge rule in order.
4. **Repeat:** Run for `num_merges` (or until target vocab size is reached).

### 3. Encoding New Text
* Replay the **exact same merge order** learned during training on new input.
* Unseen words get composed from learned subwords/bytes rather than failing outright.

### 4. Key Trade-offs & Production Impact
* **More merges / larger vocab:** Shorter token sequences, faster inference/fewer tokens per prompt, larger embedding table.
* **Fewer merges / smaller vocab:** Longer token sequences, more compute per sequence, smaller embedding matrix.
* **Token ID vs. Embedding:** BPE maps text $\rightarrow$ **token IDs** (integers); the model's embedding layer maps token IDs $\rightarrow$ **dense vectors**.
* **Special tokens:** Reserved tokens (e.g., BOS, EOS, PAD, chat/role delimiters) control boundaries and formatting.

In [16]:

from collections import Counter

# Our tiny training corpus
corpus = Counter({
    "low": 5,
    "lower": 2,
    "lowest": 1,
    "newer": 3,
    "wider": 2
})

# Step 1: Split each word into characters
# </w> marks the end of a word.

splits = {
    word: list(word) + ["</w>"]
    for word in corpus
}


# Step 2: Count adjacent pairs
def get_pair_frequencies(splits, corpus):

    pairs = Counter()

    for word, symbols in splits.items():

        for i in range(len(symbols) - 1):

            pair = (symbols[i], symbols[i + 1])

            # Weight by word frequency
            pairs[pair] += corpus[word]

    return pairs


# Step 3: Merge a selected pair
def merge_pair(pair, splits):

    new_splits = {}

    first, second = pair

    for word, symbols in splits.items():

        merged = []
        i = 0

        while i < len(symbols):

            if (
                i < len(symbols) - 1
                and symbols[i] == first
                and symbols[i + 1] == second
            ):
                merged.append(first + second)
                i += 2

            else:
                merged.append(symbols[i])
                i += 1

        new_splits[word] = merged

    return new_splits


# Step 4: Train BPE
def train_bpe(splits, corpus, num_merges):

    merges = []

    for step in range(num_merges):

        pair_freq = get_pair_frequencies(
            splits, corpus
        )

        if not pair_freq:
            break

        best_pair = max(
            pair_freq,
            key=pair_freq.get
        )

        print(
            f"Merge {step + 1}: {best_pair}, "
            f"frequency = {pair_freq[best_pair]}"
        )

        splits = merge_pair(best_pair, splits)

        merges.append(best_pair)

    return splits, merges


# Train
final_splits, merges = train_bpe(
    splits, corpus, num_merges=10
)

print("\nFinal vocabulary:")

for word, symbols in final_splits.items():
    print(word, "->", symbols)

print("\nLearned merges:")
print(merges)


Merge 1: ('l', 'o'), frequency = 8
Merge 2: ('lo', 'w'), frequency = 8
Merge 3: ('e', 'r'), frequency = 7
Merge 4: ('er', '</w>'), frequency = 7
Merge 5: ('low', '</w>'), frequency = 5
Merge 6: ('n', 'e'), frequency = 3
Merge 7: ('ne', 'w'), frequency = 3
Merge 8: ('new', 'er</w>'), frequency = 3
Merge 9: ('low', 'er</w>'), frequency = 2
Merge 10: ('w', 'i'), frequency = 2

Final vocabulary:
low -> ['low</w>']
lower -> ['lower</w>']
lowest -> ['low', 'e', 's', 't', '</w>']
newer -> ['newer</w>']
wider -> ['wi', 'd', 'er</w>']

Learned merges:
[('l', 'o'), ('lo', 'w'), ('e', 'r'), ('er', '</w>'), ('low', '</w>'), ('n', 'e'), ('ne', 'w'), ('new', 'er</w>'), ('low', 'er</w>'), ('w', 'i')]


# NOW LET'S USE A REAL WORLD TOKENIZER



```



In [17]:
pip install transformers

In [20]:
from transformers import AutoTokenizer

#just loading a pretrainde gpt-2 transformer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text="my name is saksham dubey ,i love machine learning and deep learning."
token= tokenizer.tokenize(text)

#.encode se encode kardiya
token_ids = tokenizer.encode(text)
#decode se (id->sentence) kardiya
decoded = tokenizer.decode(token_ids)

print("Original text:", text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)
print("Number of tokens:", len(token_ids))
print("Decoded text:", decoded)

# Vocabulary info
print("Vocabulary size:", tokenizer.vocab_size)


Original text: my name is saksham dubey ,i love machine learning and deep learning.
Tokens: ['I', 'love', 'machine', 'learning']
Token IDs: [1820, 1438, 318, 264, 461, 1477, 321, 10873, 2959, 837, 72, 1842, 4572, 4673, 290, 2769, 4673, 13]
Number of tokens: 18
Decoded text: my name is saksham dubey ,i love machine learning and deep learning.
Vocabulary size: 50257


In [21]:
texts = [
    "Hello",
    "Hello world",
    "Hello, world!",
    "Artificial Intelligence",
    "ArtificialIntelligence",
    "unbelievable",
    "unbelievability"
]

for text in texts:
    ids = tokenizer.encode(text)

    print(text)
    print("Tokens:", tokenizer.tokenize(text))
    print("Token count:", len(ids))
    print("-" * 40)

Hello
Tokens: ['Hello']
Token count: 1
----------------------------------------
Hello world
Tokens: ['Hello', 'Ġworld']
Token count: 2
----------------------------------------
Hello, world!
Tokens: ['Hello', ',', 'Ġworld', '!']
Token count: 4
----------------------------------------
Artificial Intelligence
Tokens: ['Art', 'ificial', 'ĠIntelligence']
Token count: 3
----------------------------------------
ArtificialIntelligence
Tokens: ['Art', 'ificial', 'Int', 'elligence']
Token count: 4
----------------------------------------
unbelievable
Tokens: ['un', 'bel', 'iev', 'able']
Token count: 4
----------------------------------------
unbelievability
Tokens: ['un', 'bel', 'iev', 'ability']
Token count: 4
----------------------------------------


Why this matters in production:
- API costs are often based on token counts.
- Context windows are measured in tokens.
- Longer token sequences require more computation.
- Tokenization affects how much information fits into a prompt.